# SKB-Q prospective Ridge allocation — code-only release

Original pre-outcome allocations are stored in a separate frozen decision JSON file, not in this public notebook. Set `SKBQ_DECISION_MANIFEST` before execution. This is a code release only; no data, results, or model weights are redistributed.


In [ ]:
import sys,subprocess,importlib.metadata as meta
wanted={'transformers':'4.49.0','datasets':'3.5.1','huggingface-hub':'0.36.2','safetensors':'0.5.3'}
missing=[]
for pkg,version in wanted.items():
    try: now=meta.version(pkg)
    except meta.PackageNotFoundError: now=None
    if now!=version:missing.append(f'{pkg}=={version}')
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
    print('Pinned dependencies installed. Restart runtime if a package was already imported with another version.')
else: print('Dependencies pinned')

In [ ]:
import os, json, hashlib, shutil, math, gc, random, platform
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

RUN_MODE='FULL'  # DRY_RUN checks model + dataset without computing new losses
assert RUN_MODE in ('FULL','DRY_RUN')
OUT=Path('/kaggle/working/SKBQ_Qwen_Prospective_Decision_FULL') if Path('/kaggle/working').exists() else Path('/mnt/data/SKBQ_Qwen_Prospective_Decision_FULL')
OUT.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(), 'Activate GPU T4 in Kaggle Notebook Settings'
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False

def canon(obj):return json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode('utf8')
def sha(obj):return hashlib.sha256(obj).hexdigest()
def save(name,data):
    path=OUT/name; tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf8')
    tmp.replace(path)
print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__,'transformers:',__import__('transformers').__version__)

In [ ]:
# The original frozen decision manifest (including maps) is not published here.
FROZEN_MANIFEST_PATH=Path(os.environ.get('SKBQ_DECISION_MANIFEST','/kaggle/input/skbq-private/FROZEN_DECISION_MANIFEST.json'))
if not FROZEN_MANIFEST_PATH.exists():
    raise FileNotFoundError('Supply the original frozen decision JSON via SKBQ_DECISION_MANIFEST')
FROZEN=json.loads(FROZEN_MANIFEST_PATH.read_text(encoding='utf-8'))
manifest_sha=FROZEN['manifest_sha256']
assert sha(canon({k:v for k,v in FROZEN.items() if k!='manifest_sha256'}))==manifest_sha
assert len(FROZEN['maps'])==7 and len({tuple(v) for v in FROZEN['maps'].values()})==7
assert all(len(v)==len(set(v))==14 and min(v)>=0 and max(v)<28 for v in FROZEN['maps'].values())
assert FROZEN['status']=='MAPS_AND_ANALYSIS_PLAN_FROZEN_BEFORE_NEW_WIKITEXT_GPU_LOSSES'
save('FROZEN_MANIFEST_COPIED_BEFORE_EVALUATION.json',FROZEN)
print('Frozen candidate manifest verified:',manifest_sha)

## 1. Load the exact Qwen model and calculate the exact 4/8-bit weight budget

Model revision and the pre-quantization embedding fingerprint are checked against the previously executed Qwen experiment. Any mismatch is a hard stop. We clone the unmodified FP16 Linear weights to CPU and restore each policy from them; policies never accumulate quantization errors.

In [ ]:
MODEL=FROZEN['model']; PIN=MODEL['revision'];MODEL_ID=MODEL['repo_id']
tok=AutoTokenizer.from_pretrained(MODEL_ID,revision=PIN,trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=PIN,torch_dtype=torch.float16,
                       attn_implementation='sdpa',trust_remote_code=False,low_cpu_mem_usage=True).eval().to('cuda')
model.config.use_cache=False
cfg=model.config
assert cfg.model_type=='qwen2' and cfg.num_hidden_layers==28 and cfg.hidden_size==1536
assert cfg.num_attention_heads==12 and cfg.num_key_value_heads==2
if model.lm_head.weight.data_ptr()==model.get_input_embeddings().weight.data_ptr():
    model.lm_head.weight=nn.Parameter(model.lm_head.weight.detach().clone(),requires_grad=False)
assert model.lm_head.weight.data_ptr()!=model.get_input_embeddings().weight.data_ptr()
embedding_pre=sha(model.get_input_embeddings().weight.detach().cpu().numpy().tobytes())
assert embedding_pre==MODEL['input_embedding_sha256_before_quantization'], 'Embedding fingerprint mismatch: STOP'
modules={n:m for n,m in model.named_modules() if isinstance(m,nn.Linear)}
expected={f'model.layers.{i}.self_attn.{p}' for i in range(28) for p in ('q_proj','k_proj','v_proj','o_proj')}
expected|={f'model.layers.{i}.mlp.{p}' for i in range(28) for p in ('gate_proj','up_proj','down_proj')}
expected.add('lm_head')
assert set(modules)==expected, 'Unexpected Linear inventory'
WEIGHT_COUNT=sum(m.weight.numel() for m in modules.values())
GROUP_WEIGHT=sum(modules[f'model.layers.0.self_attn.{p}'].weight.numel() for p in ('q_proj','k_proj','v_proj','o_proj'))
assert all(sum(modules[f'model.layers.{i}.self_attn.{p}'].weight.numel() for p in ('q_proj','k_proj','v_proj','o_proj'))==GROUP_WEIGHT for i in range(28))
BUDGET=4+4*14*GROUP_WEIGHT/WEIGHT_COUNT
assert WEIGHT_COUNT==MODEL['accounted_linear_params'] and GROUP_WEIGHT==MODEL['attention_group_params']
assert abs(BUDGET-MODEL['avgbit_linear'])<1e-12
master={n:m.weight.detach().cpu().clone() for n,m in modules.items()}
save('MODEL_AND_BUDGET_LOCK.json',{'model_id':MODEL_ID,'model_sha':PIN,'torch':torch.__version__,
    'transformers':__import__('transformers').__version__,'group_params':GROUP_WEIGHT,
    'linear_params':WEIGHT_COUNT,'avgbit_linear':BUDGET,'embedding_sha256':embedding_pre,
    'manifest_sha256':manifest_sha,'linear_module_names':sorted(modules)})
print('Matched mean-bit proxy:',repr(BUDGET),'modules:',len(modules))

## 2. Load *new* frozen WikiText-2 test split

Dataset revision, configuration and split are immutable in the pre-evaluation manifest. The first 64 blocks are fixed; no data filtering or test-set selection based on evaluation outcomes is allowed. Token IDs are hashed and recorded **before running any policy**.

In [ ]:
EV=FROZEN['target_corpus']
ds=load_dataset(EV['id'],EV['config'],revision=EV['revision'],split=EV['split'])
texts=[str(row['text']).strip() for row in ds if row['text'] and str(row['text']).strip()]
ids=tok('\n\n'.join(texts),add_special_tokens=False)['input_ids']
COUNT=EV['nonoverlapping_blocks'];L=EV['seq_length'];BEGIN=EV['block_start_index']
assert COUNT==64 and L==1024 and BEGIN==0
need=(BEGIN+COUNT)*L
assert len(ids)>=need,f'Dataset too short: {len(ids)} < {need} tokens; do not change the number of blocks'
blocks=np.asarray(ids[BEGIN*L:need],dtype='<i4').reshape(COUNT,L)
token_sha=sha(blocks.tobytes())
save('TARGET_TOKEN_LOCK_BEFORE_EVALUATION.json',{'model_sha':PIN,
    'dataset_id':EV['id'],'config':EV['config'],'revision':EV['revision'],'split':EV['split'],
    'used_rows':len(texts),'available_tokens':len(ids),'evaluated_tokens':COUNT*L,
    'predicted_tokens':COUNT*(L-1),'block_range':[BEGIN,BEGIN+COUNT-1],
    'target_token_sha256':token_sha,'manifest_sha256':manifest_sha})
print('64 fixed test blocks locked:',token_sha)
if RUN_MODE=='DRY_RUN':
    print('DRY_RUN complete: no model NLL outcomes evaluated; switch RUN_MODE to FULL and Run All')

## 3. Evaluate all seven frozen policies (FULL)

**Resume support:** checkpoints are saved every four blocks. If restarting in the same persistent working directory, a checkpoint is accepted only when its fingerprint matches model, map, test tokens, mode and frozen manifest. Otherwise the run stops.

In [ ]:
def rtn_row_chunk(w,bits):
    qmax=(1<<(bits-1))-1
    v=w.to(device='cuda',dtype=torch.float32)
    scale=v.abs().amax(dim=1,keepdim=True)/qmax
    scale=torch.where(scale>0,scale,torch.ones_like(scale))
    return ((v/scale).round().clamp(-qmax,qmax)*scale).to(torch.float16)

def map_bits(selected):
    assert len(selected)==14
    b={n:4 for n in modules}
    for layer in selected:
        for p in ('q_proj','k_proj','v_proj','o_proj'):
            b[f'model.layers.{layer}.self_attn.{p}']=8
    avg=sum(modules[n].weight.numel()*b[n] for n in modules)/WEIGHT_COUNT
    assert abs(avg-BUDGET)<1e-12
    return b,avg

def install_bits(mapping):
    with torch.no_grad():
        for n,m in modules.items():
            original=master[n];bit=mapping[n]
            for start in range(0,original.shape[0],4096):
                end=min(start+4096,original.shape[0]);m.weight.data[start:end].copy_(rtn_row_chunk(original[start:end],bit))

ORDER=list(FROZEN['maps'])
RESULTS=[]
if RUN_MODE=='FULL':
    for pidx,policy in enumerate(ORDER,1):
        selected=FROZEN['maps'][policy]
        mapping,bit=map_bits(selected)
        signature=sha(canon({'model_revision':PIN,'policy':policy,'map':selected,'target_sha':token_sha,
                             'manifest_sha':manifest_sha,'avgbit':bit,'blocks':COUNT}))
        path=OUT/f'checkpoint_{policy}.json';losses=[]
        if path.exists():
            prior=json.loads(path.read_text())
            assert prior['fingerprint']==signature,'Stale checkpoint: STOP'
            losses=list(map(float,prior['block_mean_nll']))
        assert len(losses)<=COUNT
        print(f'[{pidx}/7] {policy}, existing {len(losses)}/{COUNT}',flush=True)
        install_bits(mapping)
        for bi in range(len(losses),COUNT):
            x=torch.as_tensor(blocks[bi],dtype=torch.long,device='cuda').unsqueeze(0)
            with torch.inference_mode():
                out=model(input_ids=x,use_cache=False)
                logits=out.logits[:,:-1,:].float().contiguous()
                loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),x[:,1:].reshape(-1))
                value=float(loss.item())
            assert math.isfinite(value)
            losses.append(value)
            del x,out,logits,loss
            if (bi+1)%4==0 or bi==COUNT-1:
                save(f'checkpoint_{policy}.json',{'policy':policy,'selected_layers':selected,'avgbit_linear':bit,
                    'model_revision':PIN,'frozen_manifest_sha256':manifest_sha,'target_token_sha256':token_sha,
                    'allocation_sha256':sha(canon(mapping)),'fingerprint':signature,
                    'block_mean_nll':losses,'blocks_complete':len(losses)})
                print(' ',bi+1,'/',COUNT,'meanNLL',round(float(np.mean(losses)),7),flush=True)
        RESULTS.append({'policy':policy,'mean_nll':float(np.mean(losses)),
            'ppl':float(math.exp(float(np.mean(losses)))),'blocks':COUNT,
            'predicted_tokens':COUNT*(L-1),'avgbit_linear':bit,'allocation_sha256':sha(canon(mapping))})
        save('PROGRESS_SUMMARY.json',RESULTS)
        torch.cuda.empty_cache()
    assert len(RESULTS)==len(ORDER)==7
    assert sha(model.get_input_embeddings().weight.detach().cpu().numpy().tobytes())==embedding_pre
    save('POLICY_RESULTS.json',RESULTS)
    print('Seven policies evaluated. Checkpoints complete.')

## 4. Execute the **predeclared**, paired analysis

The first two comparisons were frozen as the primary test before computing new losses. Report positive or negative outcomes without changing the candidate set. Bootstrap resamples **token blocks** rather than documents; uncertainty is descriptive because adjacent blocks are not independent documents.

In [ ]:
if RUN_MODE=='FULL':
    losses={}
    for policy in ORDER:
        checkpoint=json.loads((OUT/f'checkpoint_{policy}.json').read_text())
        assert checkpoint['frozen_manifest_sha256']==manifest_sha
        assert checkpoint['target_token_sha256']==token_sha
        assert checkpoint['blocks_complete']==COUNT
        losses[policy]=np.asarray(checkpoint['block_mean_nll'],dtype=np.float64)
        assert len(losses[policy])==64 and np.isfinite(losses[policy]).all()
    rng=np.random.default_rng(20261017)
    boot_idx=rng.integers(0,COUNT,size=(5000,COUNT))
    ridge=losses['Ridge_TopK']
    reference={
      'BestDev_R21':losses['BestDev_R21'],
      'mean_of_three_locked_random':np.mean(np.stack([losses[f'Random_{j:02d}'] for j in range(1,4)]),axis=0),
      'Late':losses['Late'],
      'OLS_TopK':losses['OLS_TopK']}
    paired={}
    for name,baseline in reference.items():
        diff=ridge-baseline
        bootstrap=diff[boot_idx].mean(axis=1)
        paired[name]={'ridge_minus_comparator_mean_nll':float(diff.mean()),
            'bootstrap_block_percentile_CI95':[float(np.quantile(bootstrap,.025)),float(np.quantile(bootstrap,.975))],
            'blocks':COUNT,'interpretation':'Negative difference favours Ridge; block bootstrap descriptive only.'}
    rule=FROZEN['frozen_decision_rule']
    pass_primary=paired['BestDev_R21']['ridge_minus_comparator_mean_nll']<0
    pass_co=paired['mean_of_three_locked_random']['ridge_minus_comparator_mean_nll']<0
    report={'study_id':FROZEN['study_id'],'manifest_sha256':manifest_sha,'model_revision':PIN,
            'target_token_sha256':token_sha,'all_policy_results':RESULTS,'paired_contrasts':paired,
            'pragmatic_success_both_mean_differences_negative':bool(pass_primary and pass_co),
            'primary_pass':bool(pass_primary),'co_primary_pass':bool(pass_co),
            'cautions':['Cross-corpus WikiText-test evidence only','Full LM head and MLP fixed at fake 4bit RTN',
                      'Bootstrap blocks may be serially dependent','No strong quantizer or hardware speed claim',
                      'One Qwen model only; frozen 30 training labels were obtained from C4 development']}
    save('PREDECLARED_DECISION_ANALYSIS.json',report)
    for n,stat in paired.items():print('Ridge minus',n,round(stat['ridge_minus_comparator_mean_nll'],7),stat['bootstrap_block_percentile_CI95'])
    print('Locked decision pass?',report['pragmatic_success_both_mean_differences_negative'])

In [ ]:
if RUN_MODE=='FULL':
    archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
    print('Download and send:',archive)
else:
    print('DRY_RUN complete, no test losses used. Change RUN_MODE to FULL and Run All to produce result ZIP.')

## Interpretation and delivery

Upload `SKBQ_Qwen_Prospective_Decision_FULL.zip`. Internal checksum checks, all seven checkpoints and the predeclared paired analysis can then be audited against the previously frozen manifest. A successful experiment would show **prospective within-Qwen choice utility on one new corpus**, not a novel additive estimator, universal transfer, or deployment-ready compression. An unsuccessful experiment is still reportable scientific evidence; do not rerank, resample or switch corpus after seeing outcomes.